# Model 2 – Load Facts to Supabase

In [13]:
!pip -q install google-genai supabase

In [14]:
import time, requests
import numpy as np, pandas as pd
from google import genai
from google.genai import types
from google.colab import userdata
from supabase import create_client

gem = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))
URL, KEY = userdata.get("SUPABASE_URL"), userdata.get("SUPABASE_SERVICE_KEY")
sb = create_client(URL, KEY)
facts = pd.read_csv("fact_bank.csv")
scores = pd.read_csv("livelihood_scores.csv")
print(len(facts), "facts;", len(scores), "score rows")

871 facts; 225 score rows


## Table columns

In [18]:
defs = requests.get(f"{URL}/rest/v1/", headers={"apikey": KEY, "Authorization": f"Bearer {KEY}"}).json()["definitions"]
cols = {t: set(defs[t]["properties"]) for t in ("livelihood_facts", "livelihood_scores")}
for t, c in cols.items(): print(t, sorted(c))

def clean(df, table):
    keep = [c for c in df.columns if c in cols[table]]
    dropped = [c for c in df.columns if c not in cols[table]]
    if dropped: print(f"{table}: columns not in table, skipped ->", dropped)
    out = df[keep].astype(object).where(df[keep].notna(), None)
    return out.to_dict("records")

livelihood_facts ['ci_high', 'ci_low', 'district', 'embedding', 'evidence_type', 'fact_id', 'id', 'is_estimated', 'livelihood', 'n_obs', 'scope', 'small_group', 'source', 'statement', 'swot', 'value']
livelihood_scores ['area', 'confidence', 'district', 'id', 'livelihood', 'market', 'profitability', 'resilience', 'viability', 'viability_high', 'viability_low', 'weights']


## Embeddings

In [16]:
import os, pickle
def embed(texts, task="RETRIEVAL_DOCUMENT", cache=None, size=20, pause=15):
    done = pickle.load(open(cache, "rb")) if cache and os.path.exists(cache) else []
    for i in range(len(done), len(texts), size):
        batch = texts[i:i + size]
        for attempt in range(10):
            try:
                r = gem.models.embed_content(model="gemini-embedding-001", contents=batch,
                    config=types.EmbedContentConfig(output_dimensionality=768, task_type=task))
                done += [np.array(e.values) for e in r.embeddings]; break
            except Exception as err:
                print("retry:", str(err)[:150]); time.sleep(30 * (attempt + 1))
        else:
            raise RuntimeError("embedding failed")
        if cache: pickle.dump(done, open(cache, "wb"))
        print(len(done), "/", len(texts), end="\r"); time.sleep(pause)
    return [(v / np.linalg.norm(v)).tolist() for v in done]

texts = (facts["livelihood"].fillna("") + " | " + facts["district"].fillna("") + " | " + facts["statement"]).tolist()
vecs = embed(texts, cache="vecs.pkl")
print(len(vecs), "embeddings,", len(vecs[0]), "dims")

retry: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more info
retry: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more info


KeyboardInterrupt: 

## Load
Clears the two tables first, so re-running is safe.

In [19]:
facts["n_obs"] = facts["n"]
facts["small_group"] = (facts["n"] < 10) & (facts["evidence_type"] == "monitoring")
scores["viability_low"] = scores["score_p5"]
scores["viability_high"] = scores["score_p95"]
scores["confidence"] = scores["rank_stability"]
facts["livelihood"] = facts["livelihood"].fillna("ALL")
sb.table("livelihood_facts").delete().neq("scope", "__none__").execute()
sb.table("livelihood_scores").delete().neq("livelihood", "__none__").execute()

rows = clean(facts, "livelihood_facts")
for r, v in zip(rows, vecs): r["embedding"] = v
for i in range(0, len(rows), 100):
    sb.table("livelihood_facts").insert(rows[i:i + 100]).execute()

srows = clean(scores, "livelihood_scores")
for i in range(0, len(srows), 100):
    sb.table("livelihood_scores").insert(srows[i:i + 100]).execute()
print("loaded")

livelihood_facts: columns not in table, skipped -> ['n']
livelihood_scores: columns not in table, skipped -> ['district_evidence', 'n_livelihood_visits', 'n_district_visits', 'score_p5', 'score_p95', 'score_range', 'rank_stability', 'rank_in_district']
loaded


## Check

In [20]:
n1 = sb.table("livelihood_facts").select("fact_id", count="exact").limit(1).execute().count
n2 = sb.table("livelihood_scores").select("livelihood", count="exact").limit(1).execute().count
print("facts in table:", n1, "| scores in table:", n2)

q = embed(["What are the weaknesses of agriculture in Ampara?"], "RETRIEVAL_QUERY")[0]
hits = sb.rpc("match_livelihood_facts", {"query_embedding": q, "p_livelihood": "AGRICULTURE & PLANTATION CROPS",
                                         "p_district": "Ampara", "match_count": 5}).execute().data
for h in hits: print(round(h["similarity"], 3), "|", h["statement"])
print("LOAD OK" if n1 == len(facts) and n2 == len(scores) and hits else "CHECK FAILED")

facts in table: 871 | scores in table: 225
0.77 | Agriculture & Plantation Crops in Ampara: 16% of visits record: storage or transport problem (95% CI 7-32%, 32 visits).
0.763 | Agriculture & Plantation Crops in Ampara: 14% of visits record: equipment problem (95% CI 5-31%, 29 visits).
0.759 | Agriculture & Plantation Crops in Ampara: 14% of visits record: inputs unavailable (95% CI 5-31%, 29 visits).
0.758 | Agriculture & Plantation Crops in Ampara: 7% of visits record: high input cost (95% CI 2-22%, 29 visits).
0.758 | Agriculture & Plantation Crops in Ampara: viability 56/100 (range 54-59); profitability 52, market 33, area 59, resilience 74; rank 9 of 9 in this district.
LOAD OK


In [17]:
print(len(vecs), len(vecs[0]))

871 768


In [21]:
print(sb.table("livelihood_facts").select("swot").neq("swot", "none").limit(3).execute().data)
print(sb.table("livelihood_scores").select("viability,viability_low,viability_high,confidence").limit(2).execute().data)

[{'swot': 'Weakness'}, {'swot': 'Weakness'}, {'swot': 'Weakness'}]
[{'viability': 57.4, 'viability_low': 55.0, 'viability_high': 59.5, 'confidence': '0.739'}, {'viability': 61.2, 'viability_low': 58.2, 'viability_high': 64.0, 'confidence': '1.0'}]
